# Suite PYR — The Python RegExp dialect

The regular expressions of Python's `re` module, a grammar of their own (the Library Reference, `re`, "Regular
Expression Syntax"): the dialect `Expressions.Python.RegExp` of the Python family, its kinds the constructs `re`'s
syntax lists, named as regexpp's where ECMAScript has them too, its `Term` of them alone, its holder, its checks, as
`re`'s parser makes them, its editions, its printing, its evaluation, which refuses each kind by name, and its
completeness against the codes `re`'s parser writes (mbse-basis [CAP-281: Python's, ECMAScript's and C++'s regular expressions as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-281.md)).

In [ ]:
import functools
import json
import re
import sys
from pathlib import Path

from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Patterns import Expressions as Patterns
from mbse.Expressions.Python import Patterns as PythonPatterns, Python3, RegExp as PythonRegExp, Types as PythonTypes
from mbse.Expressions.Python.Python3 import Evaluators as PV, Expressions as P, Text as PText
from mbse.Expressions.Python.Patterns import Expressions as PP
from mbse.Expressions.Python.RegExp import Expressions as R, Text as RText
from mbse.Expressions.Python.Types import Expressions as PT
from mbse.Expressions.Framework import Domains as D, Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators, YAML
from support import raises

store = Builders.Store()  # the framework, the standard dialects Python3's complex numbers use, and the Python family
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, Patterns.install, Python3.install,
                PythonTypes.install, PythonPatterns.install, PythonRegExp.install]:
    store.transaction(install)
store.transaction(PythonRegExp.define)  # the versions the RegExp dialect covers
Term = store.registered("Expressions.Framework.Term")


def held(spec, dialect=R.DIALECT):
    """The holder of the term `spec` writes, `dialect`'s own (the RegExp dialect's unless given), written in a
    transaction."""
    with store.transaction():
        holders = functools.reduce(getattr, dialect.path.split("."), store.Expressions)
        return holders.Expression(lambda e: e.root(dialect.spec(spec)))


def repeat(element, least=None, most=None, greedy=None, possessive=None):
    """A quantifier of `element`, its bounds and its options where given."""
    def write(q):
        for name, value in (("min", least), ("max", most), ("greedy", greedy), ("possessive", possessive)):
            if value is not None:
                getattr(q, name)(value)
        return q.element(element) if element is not None else q
    return lambda u: u.quantifier(write)


regexp = lambda spec: F.term(held(spec))  # noqa: E731: a term of the RegExp dialect, held by its holder
char = lambda value, raw=None: lambda u: u.character(lambda c: c.value(value) if raw is None else c.raw(raw).value(value))  # noqa: E731
seq = lambda *elements: lambda u: u.alternative(lambda a: a.elements(list(elements)))  # noqa: E731: an alternative
alts = lambda *alternatives: lambda u: u.pattern(lambda p: p.alternatives(list(alternatives)))  # noqa: E731: a pattern
group = lambda *alternatives: lambda u: u.group(lambda g: g.alternatives(list(alternatives)))  # noqa: E731: `(?:...)`
capture = lambda *alternatives, name=None: lambda u: u.capturinggroup(  # noqa: E731: `(...)`, `(?P<name>...)`
    lambda g: (g if name is None else g.name(name)).alternatives(list(alternatives)))
atomic = lambda *alternatives: lambda u: u.atomicgroup(lambda g: g.alternatives(list(alternatives)))  # noqa: E731: `(?>...)`
klass = lambda *elements, negate=None: lambda u: u.characterclass(  # noqa: E731: `[...]`, `[^...]`
    lambda c: (c if negate is None else c.negate(negate)).elements(list(elements)))
span = lambda low, high: lambda u: u.characterclassrange(lambda r: r.min(low).max(high))  # noqa: E731: `a-z`
charset = lambda kind, negate=None: lambda u: u.characterset(  # noqa: E731: `.`, `\d`, `\D`
    lambda s: s.setKind(kind) if negate is None else s.setKind(kind).negate(negate))
check = lambda kind, *alternatives, negate=None: lambda u: u.assertion(  # noqa: E731: `^`, `\b`, `(?=...)`
    lambda a: (a.assertionKind(kind) if negate is None else a.assertionKind(kind).negate(negate)).alternatives(list(alternatives)))
ref = lambda key: lambda u: u.backreference(lambda b: b.ref(key))  # noqa: E731: `\\1`, `(?P=name)`
cond = lambda key, yes, no=None: lambda u: u.conditional(  # noqa: E731: `(?(1)yes|no)`
    lambda c: c.ref(key).yes(yes) if no is None else c.ref(key).yes(yes).no(no))
flags = lambda add, remove="", *alternatives: lambda u: u.flags(  # noqa: E731: `(?i)`, `(?i-m:...)`
    lambda f: (f.add(add) if add else f).remove(remove).alternatives(list(alternatives)) if remove else
    (f.add(add) if add else f).alternatives(list(alternatives)))
comment = lambda text=None: lambda u: u.comment(lambda c: c if text is None else c.text(text))  # noqa: E731: `(?#...)`
a, b = char(97), char(98)

<a id="PYR-01"></a>
## PYR-01 · The RegExp dialect's union, its kinds' schemas named as regexpp's, and its holder: validated, linked by `HasRoot`, round-tripped through JSON and YAML; read by each dialect of its family

In [ ]:
# Its kinds: the constructs `re`'s syntax lists, named and shaped as regexpp's where ECMAScript has them too, then
# Python's own and its comment; its Term lists them and invalid syntax alone, and it has no helper of its own.
union = lambda dialect: [b.name for b in store.registered(dialect.union).branches]  # noqa: E731: its Term's branches
assert R.TAGS == ("pattern", "alternative", "group", "capturinggroup", "quantifier", "characterclass", "characterclassrange",
                  "characterset", "character", "assertion", "backreference", "conditional", "atomicgroup", "flags",
                  "comment") and R.HELPERS == ()
assert union(R.DIALECT) == [*R.TAGS, "error", "missing"] and list(R.DIALECT.kinds()) == [*R.TAGS, "error", "missing"]
assert not set(R.TAGS) & {tag for d in (P.DIALECT, PT.DIALECT, PP.DIALECT) for tag in [*union(d), *d.kinds()]}  # its tags its own
assert R.DIALECT.name() == "Python RegExp" and R.DIALECT.tag == "python_regexp"
assert R.DIALECT.path == "Python.RegExp" and R.DIALECT.union == "Expressions.Python.RegExp.Term"
RegExpTerm = store.registered(R.DIALECT.union)
kinds = {b.name: b.schema for b in RegExpTerm.branches}
assert [kinds[tag].name for tag in R.TAGS] == [f"Expressions.Python.RegExp.{kind}" for kind in (
    "Pattern", "Alternative", "Group", "CapturingGroup", "Quantifier", "CharacterClass", "CharacterClassRange",
    "CharacterSet", "Character", "Assertion", "Backreference", "Conditional", "AtomicGroup", "Flags", "Comment")]
assert {tag: list(kinds[tag].properties) for tag in R.TAGS} == {  # each a value object, its properties regexpp's names
    "pattern": ["alternatives"], "alternative": ["elements"], "group": ["alternatives"],
    "capturinggroup": ["name", "alternatives"], "quantifier": ["min", "max", "greedy", "possessive", "element"],
    "characterclass": ["negate", "elements"], "characterclassrange": ["min", "max"], "characterset": ["setKind", "negate"],
    "character": ["raw", "value"], "assertion": ["assertionKind", "negate", "alternatives"], "backreference": ["ref"],
    "conditional": ["ref", "yes", "no"], "atomicgroup": ["alternatives"],
    "flags": ["add", "remove", "alternatives"], "comment": ["text"]} and not any(kinds[t].ref for t in R.TAGS)
assert all(kinds[tag].properties[name].schema.element is RegExpTerm for tag, name in [
    ("pattern", "alternatives"), ("alternative", "elements"), ("group", "alternatives"), ("capturinggroup", "alternatives"),
    ("characterclass", "elements"), ("assertion", "alternatives"), ("atomicgroup", "alternatives"), ("flags", "alternatives")])
assert all(kinds[tag].properties[name].schema is RegExpTerm for tag, name in [
    ("quantifier", "element"), ("characterclassrange", "min"), ("characterclassrange", "max"), ("conditional", "yes"),
    ("conditional", "no")])
assert F._parts(kinds["quantifier"]) == (("element",), (), ("min", "max", "greedy", "possessive"))
assert F._parts(kinds["conditional"]) == (("yes", "no"), (), ("ref",))
assert F._parts(kinds["flags"]) == ((), ("alternatives",), ("add", "remove")) and F._parts(kinds["character"]) == ((), (), ("raw", "value"))
# No subterm of its family's other dialects is of its Term: a pattern is a string in Python3, which no reader guesses
# is a regular expression (PRI-025); mbse-programs, or a constraint's author, holds one through its holder. Its Term is a
# branch of the framework's, its dialect a singleton.
holds = lambda schema: any(RegExpTerm in (p.schema, getattr(p.schema, "element", None)) for p in schema.properties.values())  # noqa: E731
assert not any(holds(b.schema) for d in (P.DIALECT, PT.DIALECT, PP.DIALECT) for b in store.registered(d.union).branches)
assert {b.name: b.schema for b in Term.branches}["python_regexp"] is RegExpTerm
assert store.singleton("Expressions.Python.RegExp") is store.Instances.Expressions.Python.RegExp
# Each dialect of the family reads its kinds, and it reads theirs: each reads the other three.
assert R.DIALECT.reads() == (P.DIALECT, PT.DIALECT, PP.DIALECT)
assert all(R.DIALECT in d.reads() for d in (P.DIALECT, PT.DIALECT, PP.DIALECT))
assert P.DIALECT.kind_named("quantifier") is R.DIALECT.kinds()["quantifier"] and R.DIALECT.kind_named("binop") is P.DIALECT.kinds()["binop"]
# Its holder (Q2, "(a)"): a reference object refining the framework's Expression, its root narrowed to its Term; it
# validates, HasRoot links it, and it round-trips through JSON and YAML, as its root does as a value of its Term.
Holder = store.registered("Expressions.Python.RegExp.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is RegExpTerm and Holder.validate() == [] and list(Holder.adjacencies) == ["abi"]
every = alts(  # (?P<n>a\d+)(?P=n)|[^a-z_]\b
    seq(capture(seq(a, repeat(charset("digit"), 1)), name="n"), ref("n")),
    seq(klass(span(a, char(122)), char(95), negate=True), check("word")))
with store.transaction():
    holder = store.Expressions.Python.RegExp.Expression(lambda e: e.root(R.DIALECT.spec(every)))
    linked = store.Schemas.OfExpression(lambda s: s.name("pyr.every").expression(lambda a: a.insert(lambda e: e.expression(holder))))
assert Validators.Validate(store)(Holder, holder) == [] and R.DIALECT.validate(holder) == []
assert [F.same(entry.expression, holder) for entry in Builders.OfExpression.Builder(linked, False, store).expression] == [True]
SOURCE = "(?P<n>a\\d+)(?P=n)|[^a-z_]\\b"
assert F.dialect_of(F.term(holder)) == R.DIALECT.union and RText.ToText(holder) == SOURCE
for write, read in [(JSON.ToJSON(store), JSON.FromJSON(store)), (YAML.ToYAML(store), YAML.FromYAML(store))]:
    assert F.same(read(Holder, write(Holder, holder)).root, holder.root)
    assert F.same(read(RegExpTerm, write(RegExpTerm, F.term(holder))), F.term(holder))
# The schema refuses a regular expression where a union that lacks it is held: Python3's holder's root.
with raises(Exception, match="'pattern' is not a member of a union value"):
    with store.transaction():
        store.Expressions.Python.Python3.Expression(lambda e: e.root(R.DIALECT.make(F.Form("pattern", {}, ()))))
# Another family's term is refused, as before.
with raises(ValueError, match="a Patterns let is not a Python RegExp term"):
    R.DIALECT.kind(F.term(held(Patterns.Expressions.let_("a", 1, Patterns.Expressions.variable("a")), Patterns.Expressions.DIALECT)))

<a id="PYR-02"></a>
## PYR-02 · Each kind and its checks, as `re`'s parser makes them, with its messages: what each kind may hold, a choice among a few, a quantifier's bounds and element, a range's order, a reference's group, an inline flags' letters

In [ ]:
# Each kind's checks: what it may hold (an alternative where alternatives stand; no pattern, alternative or range
# among an alternative's elements; a character, a range or a set within a class; a character at either end of a range;
# an element a quantifier can repeat), a choice among a few (a set's and an assertion's kind), and its attributes (a
# quantifier's bounds and options, a reference's group, a group's name, an inline flags' letters, a comment's text),
# each with `re`'s parser's message where it has one. A backreference out of a pattern is not resolved (PYR-04).
validate = lambda spec: R.DIALECT.validate(regexp(spec))  # noqa: E731
for spec, problems in [
    (alts(seq(a, group(seq(b), seq()))), []), (seq(), []),
    (alts(a), ["a pattern's alternative cannot be a character"]),
    (group(seq(a), b), ["a group's alternative cannot be a character"]),
    (capture(a, name="x"), ["a capturinggroup's alternative cannot be a character"]),
    (atomic(seq(a)), []), (atomic(alts(seq())), ["an atomicgroup's alternative cannot be a pattern"]),
    (seq(alts(seq()), seq(), span(a, b)), ["an alternative's element cannot be a pattern",
                                          "an alternative's element cannot be an alternative",
                                          "an alternative's element cannot be a characterclassrange"]),
    (klass(a, span(a, char(122)), charset("digit", True)), []),
    (klass(group(seq(a)), check("start")), ["a characterclass's element cannot be a group",
                                            "a characterclass's element cannot be an assertion"]),
    (klass(charset("any")), ["a characterclass's element cannot be the characterset any"]),
    (klass(), ["a characterclass needs an element"]),
    (span(char(122), a), ["bad character range z-a"]), (span(char(93, "\\]"), char(65)), ["bad character range \\]-A"]),
    (span(charset("digit"), a), ["a characterclassrange's min must be a character, got a characterset"]),
    (span(a, klass(b)), ["a characterclassrange's max must be a character, got a characterclass"]),
    (lambda u: u.characterclassrange(lambda r: r.min(a)), ["a characterclassrange needs a max"]),
    (charset("any"), []), (charset("word", True), []), (charset("letter"), ["CharacterSet.setKind cannot be 'letter'"]),
    (charset("any", True), ["the characterset any cannot be negated"]),
    (lambda u: u.characterset(lambda s: s.negate(True)), ["a characterset needs a setKind"]),
    (check("start"), []), (check("word", negate=True), []), (check("lookahead", seq(a)), []),
    (check("lookbehind", seq(a), seq(b), negate=True), []),
    (check("end", negate=True), ["the assertion end cannot be negated"]),
    (check("lookahead"), ["the assertion lookahead needs an alternative"]),
    (check("stringend", seq(a)), ["the assertion stringend holds no alternatives"]),
    (check("lookahead", a), ["an assertion's alternative cannot be a character"]),
    (check("ahead"), ["Assertion.assertionKind cannot be 'ahead'"]),
    (lambda u: u.assertion(lambda s: s.negate(True)), ["an assertion needs an assertionKind"]),
    (repeat(a, 0), []), (repeat(a, 2, 5, greedy=False), []), (repeat(a, 1, possessive=True), []),
    (repeat(a, 0, greedy=True, possessive=False), []), (repeat(a, 3, 3), []),
    (repeat(a, -1), ["a quantifier's min cannot be negative, got -1"]),
    (repeat(a, 2, 1), ["min repeat greater than max repeat"]),
    (repeat(a, 0, greedy=False, possessive=True), ["a quantifier is lazy or possessive, not both"]),
    (repeat(repeat(a, 0), 1), ["multiple repeat"]),
    (repeat(check("start"), 0), ["nothing to repeat"]), (repeat(check("word"), 0), ["nothing to repeat"]),
    (repeat(check("lookahead", seq(a)), 0), []), (repeat(comment("x"), 0), ["nothing to repeat"]),
    (repeat(flags("i"), 0), ["nothing to repeat"]), (repeat(flags("i", "", seq(a)), 0), []),
    (repeat(seq(a), 0), ["nothing to repeat"]),
    (repeat(group(seq(a)), 0), []), (repeat(ref(1), 1), []), (repeat(cond(1, seq(a)), 0), []),
    (repeat(a), ["a quantifier needs a min"]), (repeat(None, 0), ["a quantifier needs an element"]),
    (ref(1), []), (ref("name"), []), (ref(0), ["a backreference's ref must be positive, got 0"]),
    (ref("1n"), ["a backreference's ref must be an identifier, got '1n'"]),
    (ref(-1), ["a backreference's ref must be positive, got -1"]),
    (lambda u: u.backreference(lambda r: r), ["a backreference needs a ref"]),
    (cond(1, seq(a)), []), (cond("n", seq(a), seq()), []),
    (cond(-2, seq(a)), ["a conditional's ref must be positive, got -2"]),
    (cond(1, a), ["a conditional's yes must be an alternative, got a character"]),
    (cond(1, seq(), b), ["a conditional's no must be an alternative, got a character"]),
    (lambda u: u.conditional(lambda c: c.ref(1)), ["a conditional needs a yes"]),
    (capture(seq(a)), []), (capture(seq(a), name="a b"), ["a capturinggroup's name must be an identifier, got 'a b'"]),
    (flags("i"), []), (flags("aimsx"), []), (flags("L"), []), (flags("", "i", seq(a)), []), (flags("u", "imsx", seq()), []),
    (flags(""), ["missing flag"]), (flags("q"), ["unknown flag"]), (flags("i", "q", seq()), ["unknown flag"]),
    (flags("au"), ["bad inline flags: flags 'a', 'u' and 'L' are incompatible"]),
    (flags("i", "a", seq(a)), ["bad inline flags: cannot turn off flags 'a', 'u' and 'L'"]),
    (flags("i", "m"), ["missing :"]),
    (flags("is", "s", seq(a)), ["bad inline flags: flag turned on and off"]),
    (comment("a note"), []), (comment(), []), (comment("a)b"), ["a comment's text cannot hold ')'"]),
]:
    assert validate(spec) == problems, (problems, validate(spec))
# A kind's problems are reported where it nests, by its argument.
assert validate(alts(seq(a, repeat(b, 2, 1)))) == ["argument 0: argument 1: min repeat greater than max repeat"]
assert R.problems(F.node("character", F.kind_of(regexp(a))[1])) == []

<a id="PYR-03"></a>
## PYR-03 · A character: its code point and its spelling as written, checked against each other, and its default spelling outside a class and within one

In [ ]:
# A character is a domain representation (PRI-023): its value, a code point, beside its raw, its spelling as written,
# which `spells` checks as `re`'s parser reads one: itself, a control character's escape, a hexadecimal, an octal or a
# named escape (its name's form alone), or a backslash before ASCII punctuation.
for text, value, spelled in [
    ("a", 97, True), ("é", 0xE9, True), ("😀", 0x1F600, True), ("b", 97, False), ("\\", 92, False), ("ab", 97, False),
    ("\\n", 10, True), ("\\t", 9, True), ("\\a", 7, True), ("\\f", 12, True), ("\\v", 11, True), ("\\r", 13, True),
    ("\\b", 8, True), ("\\n", 13, False), ("\\0", 0, True), ("\\.", 46, True), ("\\-", 45, True), ("\\ ", 32, True),
    ("\\d", 100, False), ("\\q", 113, False), ("\\1", 49, False), ("\\é", 0xE9, False),
    ("\\x41", 65, True), ("\\x4", 4, False), ("\\x4g", 0x4, False), ("\\x42", 65, False), ("\\u00e9", 0xE9, True),
    ("\\u00E9", 0xE9, True), ("\\U0001f600", 0x1F600, True), ("\\U1f600", 0x1F600, False),
    ("\\N{LATIN SMALL LETTER A}", 97, True), ("\\N{}", 97, False), ("\\N{A}B}", 97, False), ("\\NA", 97, False),
    ("\\N{EM DASH", 0x2014, False), ("\\101", 65, True), ("\\012", 10, True), ("\\01", 1, True), ("\\377", 255, True),
    ("\\400", 256, False), ("\\0123", 83, False), ("\\18", 1, False), ("\\101", 66, False), ("x\\", 120, False),
]:
    assert R.spells(text, value) is spelled, (text, value)
# Without a raw, a character is written in its default spelling: printable ASCII as itself, escaped where it is
# special, outside a class (`re`'s special characters, a space and `#`, which verbose mode skips) or within one (a
# backslash, `]`, `^`, `-` and `[`); a control character by its escape or its code; any other as itself, but a
# surrogate, by its code. Each default spelling spells its value.
for value, outside, within in [
    (97, "a", "a"), (46, "\\.", "."), (93, "\\]", "\\]"), (45, "-", "\\-"), (94, "\\^", "\\^"), (91, "\\[", "\\["),
    (32, "\\ ", " "), (35, "\\#", "#"), (124, "\\|", "|"), (92, "\\\\", "\\\\"), (10, "\\n", "\\n"), (7, "\\a", "\\a"),
    (0, "\\x00", "\\x00"), (8, "\\x08", "\\x08"), (0x7F, "\\x7f", "\\x7f"), (0x85, "\\x85", "\\x85"), (0xA0, "\xa0", "\xa0"),
    (0xE9, "é", "é"), (0x2014, "—", "—"), (0xD800, "\\ud800", "\\ud800"), (0x1F600, "😀", "😀"),
]:
    assert R.spelling(value) == outside and R.spelling(value, True) == within, (value, R.spelling(value), R.spelling(value, True))
    assert R.spells(outside, value) and R.spells(within, value)
assert R.spelling(0x110000) == "\\U00110000"  # beyond Unicode, as validation refuses, by its code
# Validation checks a character's value is a code point, and its raw, where it has one, spells it.
for spec, problems in [
    (char(97), []), (char(97, "a"), []), (char(97, "\\x61"), []), (char(0x10FFFF), []),
    (char(97, "b"), ["a character's raw 'b' does not spell U+0061"]),
    (char(0x1F600, "\\u1f60"), ["a character's raw '\\\\u1f60' does not spell U+1F600"]),
    (char(-1), ["a character's value must be a code point, got -1"]),
    (char(0x110000, "x"), ["a character's value must be a code point, got 1114112"]),
    (lambda u: u.character(lambda c: c.raw("a")), ["a character needs a value"]),
    (lambda u: u.character(lambda c: c.value(True)), ["a character's value must be a int, got bool"]),
]:
    assert validate(spec) == problems, (problems, validate(spec))

<a id="PYR-04"></a>
## PYR-04 · A pattern's groups and references, checked at its root; its forms and `make`; no kind binds a name, and a tree is read by each dialect of its family

In [ ]:
# At its root, a pattern's groups are numbered in the order they open, each name once, and each backreference and
# conditional refers to a group it has; a global flags stands only at its start, before anything but comments and other
# global flags (`re`'s parser's messages). Out of a pattern, nothing is resolved.
ngroup = capture(seq(b), name="n")
for spec, problems in [
    (alts(seq(capture(seq(a)), ngroup, ref(2), ref("n"), cond(1, seq(a), seq(b)), cond("n", seq()))), []),
    (alts(seq(capture(seq(a), name="n"), ngroup)), ["redefinition of group name 'n' as group 2; was group 1"]),
    (alts(seq(ngroup, ref("m"), cond("m", seq()))), ["unknown group name 'm'", "unknown group name 'm'"]),
    (alts(seq(ngroup, ref(2)), seq(cond(3, seq()))), ["invalid group reference 2", "invalid group reference 3"]),
    (alts(seq(ngroup), seq(capture(seq(capture(seq(a)), ref(3))))), []),  # groups numbered across alternatives, nested
    (alts(seq(flags("i"), comment("c"), flags("m"), a)), []), (alts(seq(flags("x")), seq(a)), []),
    (alts(seq(a, flags("i"))), ["global flags not at the start of the expression"]),
    (alts(seq(), seq(flags("i"))), ["global flags not at the start of the expression"]),
    (alts(seq(group(seq(flags("i"))))), ["global flags not at the start of the expression"]),
    (alts(lambda u: u.missing(lambda m: m.kind("alternative")), seq(flags("i"))),
     ["global flags not at the start of the expression", "argument 0: missing alternative"]),
    (alts(seq(flags("i", "", seq(a)), a)), []), (alts(), []),
    (alts(seq(lambda u: u.backreference(lambda r: r), ref(0))),
     ["argument 0: argument 0: a backreference needs a ref",
      "argument 0: argument 1: a backreference's ref must be positive, got 0"]),
]:
    assert validate(spec) == problems, (problems, validate(spec))
# Each kind an application of no domain known statically, which the framework reads by its structure alone: its form,
# from which make writes it again, by the dialect whose union holds it.
assert all(R.DIALECT.kinds()[tag].role == F.APPLICATION for tag in R.TAGS)
samples = alts(
    seq(group(seq(a), seq()), capture(seq(repeat(klass(span(a, char(122)), charset("digit")), 1, 3, greedy=False)), name="n"),
        check("lookbehind", seq(b), negate=True), ref("n"), ref(1), cond(1, seq(a), seq(b)), atomic(seq(charset("any"))),
        repeat(b, 0, possessive=True), check("start"), char(10, "\\n"), comment("c")),
    seq(flags("i", "m", seq(a)), check("end")))
tree = regexp(samples)
walked = list(F.walk(tree))
assert {F.kind_of(node)[0] for node in walked} == set(R.TAGS) and len(walked) == 37
for node in walked:
    made = F.term(held(R.DIALECT.make(F.form(node))))
    assert F.same(made, node) and made is not node
quantified = F.form(F.form(F.form(F.form(tree).arguments[0]).arguments[1]).arguments[0]).arguments[0]
formed = F.form(quantified)
assert formed == F.Form("quantifier", {"min": 1, "max": 3, "greedy": False}, formed.arguments) and len(formed.arguments) == 1
assert F.form(regexp(cond(1, seq(a)))).arguments[1] is None and F.form(regexp(flags("i"))) == F.Form("flags", {"add": "i"}, ())
# A named group's name is a label a backreference resolves, not a name the expression binds: no kind refines Let or has
# scopes, and a tree has no free names. Every dialect of the family reads it, validates it and prints it alike.
assert not any(F.is_binder(kinds[tag]) or R.DIALECT.kinds()[tag].scopes is not None for tag in R.TAGS)
assert SY.free(tree, R.DIALECT) == frozenset() and F.name_of(regexp(ngroup)) == "n"
assert R.DIALECT.infer(tree) is D.Anything
assert all(dialect.validate(tree) == [] for dialect in (R.DIALECT, P.DIALECT, PT.DIALECT, PP.DIALECT))
assert RText.ToText is PText.ToText and PText.ToText(tree) == RText.ToText(tree)
assert RText.ToText(tree) == "(?:a|)(?P<n>[a-z\\d]{1,3}?)(?<!b)(?P=n)\\1(?(1)a|b)(?>.)b*+^\\n(?#c)|(?i-m:a)$"

<a id="PYR-05"></a>
## PYR-05 · Invalid syntax where a regular expression's term stands: validated, printed as written, and refused by evaluation

In [ ]:
# Invalid syntax: the RegExp dialect's Term holds `error` and `missing`, wherever its terms stand; validation reports
# them, printers write them as written, and evaluation refuses them by the RegExp dialect's name.
broken = regexp(alts(seq(a, lambda u: u.error(lambda e: e.text("[a-").terms([R.DIALECT.spec(b)]))),
                     lambda u: u.missing(lambda m: m.kind("alternative"))))
assert R.DIALECT.validate(broken) == ["argument 0: argument 1: invalid syntax '[a-'", "argument 1: missing alternative"]
assert RText.ToText(broken) == "a[a-|"
assert RText.ToText(regexp(seq(capture(seq(a)), ref(1), lambda u: u.missing(lambda m: m)))) == "(a)\\1"  # missing, written as nothing
assert [R.DIALECT.kind(x)[2].role for x in F.form(broken).arguments] == [F.APPLICATION, F.INVALID]
within = regexp(klass(a, lambda u: u.error(lambda e: e.text("\\")), span(lambda u: u.missing(lambda m: m), b)))
assert R.DIALECT.validate(within) == ["argument 1: invalid syntax '\\\\'", "argument 2: min: missing a term"]
assert RText.ToText(within) == "[a\\-b]"
assert R.DIALECT.validate(regexp(repeat(lambda u: u.missing(lambda m: m.kind("atom")), 0))) == ["element: missing atom"]
with raises(ValueError, match="Python RegExp error has no text to print"):
    RText.ToText(regexp(seq(a, lambda u: u.error(lambda e: e))))
with raises(ValueError, match="Python RegExp missing is not evaluated"):
    PV.OfAny(regexp(lambda u: u.missing(lambda m: m)))

<a id="PYR-06"></a>
## PYR-06 · Its editions as data: Python 3.0 to 3.15, each with its own kinds it has, `atomicgroup` from 3.11

In [ ]:
# Its editions, as data: its define covers Python 3.0 to 3.15, owned by their published names, each with its own kinds
# it has (`atomicgroup` since Python 3.11, as `re`'s parser first wrote ATOMIC_GROUP; every other since 3.0) and none
# of the kinds it reads; run again, it writes nothing.
regexp_ = store.Instances.Expressions.Python.RegExp
VERSIONS = tuple(f"Python 3.{minor}" for minor in range(16))
assert tuple(FS.standards(regexp_)) == VERSIONS and R.SINCE == {**dict.fromkeys(R.TAGS, 0), "atomicgroup": 11}
assert FS.tags(FS.find(regexp_, "Python 3.11")) == ("Python 3.11",) and FS.find(regexp_, "Python 2.7") is None
assert FS.properties(FS.find(regexp_, "Python 3.0")) == {"bodies": ("org.python",), "revision": ("version", "3.0")}
older = [kinds[tag] for tag in R.TAGS if tag != "atomicgroup"]
assert all(FS.kinds(FS.find(regexp_, f"Python 3.{minor}")) == older for minor in range(11))
assert all(FS.kinds(FS.find(regexp_, f"Python 3.{minor}")) == [kinds[tag] for tag in R.TAGS] for minor in range(11, 16))
assert tuple(FS.of(regexp_, kinds["atomicgroup"])) == VERSIONS[11:] and tuple(FS.of(regexp_, kinds["pattern"])) == VERSIONS
assert FS.of(regexp_, store.registered("Expressions.Framework.Error")) == {}
assert store.transaction(PythonRegExp.define)[0]["Python_3_11"] is FS.find(regexp_, "Python 3.11")
assert tuple(FS.standards(regexp_)) == VERSIONS  # nothing written again

<a id="PYR-07"></a>
## PYR-07 · Printing every kind as `re`'s syntax: a quantifier's shortest bounds, a character in its spelling or escaped where it stands, a numbered backreference before a digit in a group

In [ ]:
# Printing, as `re`'s syntax writes each kind: alternatives joined by `|`; a group, a named one, an atomic one, a
# lookaround and a scoped flags around their alternatives; a quantifier's bounds in their shortest form, then `?` where
# lazy and `+` where possessive; a class, a range, a set; a character in its spelling as written, else escaped as where
# it stands; an assertion; a backreference by number or by name, a numbered one before a digit within a group of its
# own; a conditional; a global flags; a comment.
cases = [
    (alts(seq(a), seq(b), seq()), "a|b|"), (seq(char(46), char(32), char(35), char(10), char(0xE9), char(0x1F600)), "\\.\\ \\#\\né😀"),
    (seq(char(97, "\\x61"), char(46, "\\."), char(0x2014, "\\N{EM DASH}")), "\\x61\\.\\N{EM DASH}"),
    (group(seq(a), seq(b)), "(?:a|b)"), (capture(seq(a)), "(a)"), (capture(seq(a), name="word"), "(?P<word>a)"),
    (atomic(seq(a, b)), "(?>ab)"),
    (repeat(a, 0), "a*"), (repeat(a, 1), "a+"), (repeat(a, 0, 1), "a?"), (repeat(a, 2, 2), "a{2}"), (repeat(a, 2), "a{2,}"),
    (repeat(a, 2, 5), "a{2,5}"), (repeat(a, 0, 5), "a{0,5}"), (repeat(a, 0, greedy=False), "a*?"),
    (repeat(a, 1, 2, greedy=False), "a{1,2}?"), (repeat(a, 1, possessive=True), "a++"), (repeat(a, 0, 1, greedy=True), "a?"),
    (repeat(group(seq(a, b)), 0, possessive=False), "(?:ab)*"),
    (klass(char(93), char(45), span(a, char(122)), charset("word"), negate=True), "[^\\]\\-a-z\\w]"),
    (klass(char(94), char(46), char(32), char(8, "\\b")), "[\\^. \\b]"), (klass(span(char(48, "\\x30"), char(57))), "[\\x30-9]"),
    (charset("any"), "."), (charset("digit", True), "\\D"), (charset("space"), "\\s"), (charset("word", True), "\\W"),
    (check("start"), "^"), (check("end"), "$"), (check("stringstart"), "\\A"), (check("stringend"), "\\Z"),
    (check("word"), "\\b"), (check("word", negate=True), "\\B"), (check("lookahead", seq(a)), "(?=a)"),
    (check("lookahead", seq(a), negate=True), "(?!a)"), (check("lookbehind", seq(a)), "(?<=a)"),
    (check("lookbehind", seq(a), seq(b), negate=True), "(?<!a|b)"),
    (seq(capture(seq(a)), ref(1)), "(a)\\1"), (seq(capture(seq(a), name="x"), ref("x")), "(?P<x>a)(?P=x)"),
    (seq(capture(seq(a)), ref(1), char(48)), "(a)(?:\\1)0"), (seq(capture(seq(a)), ref(1), repeat(char(48), 0)), "(a)(?:\\1)0*"),
    (seq(capture(seq(a)), ref(1), char(48, "\\x30")), "(a)\\1\\x30"), (seq(capture(seq(a), name="x"), ref("x"), char(48)), "(?P<x>a)(?P=x)0"),
    (seq(repeat(capture(seq(a)), 0, 1), cond(1, seq(b), seq(a))), "(a)?(?(1)b|a)"),
    (seq(repeat(capture(seq(a), name="x"), 0, 1), cond("x", seq(b))), "(?P<x>a)?(?(x)b)"),
    (seq(flags("i"), a), "(?i)a"), (flags("i", "m", seq(a)), "(?i-m:a)"), (flags("", "s", seq(a)), "(?-s:a)"),
    (flags("ax", "", seq(a), seq()), "(?ax:a|)"), (seq(comment("note"), a), "(?#note)a"), (comment(), "(?#)"),
]
for spec, source in cases:
    assert RText.ToText(regexp(spec)) == source, (source, RText.ToText(regexp(spec)))
    assert R.DIALECT.validate(regexp(spec)) == [], source
assert {F.kind_of(node)[0] for spec, _ in cases for node in F.walk(regexp(spec))} == set(R.TAGS)
assert RText.TEXTS.keys() == set(R.TAGS) and RText.CLOSED == 17
assert RText.ToText(regexp(span(char(93), char(94)))) == "\\]-\\^"  # a range alone, its ends spelled as within a class
# Python's own `re` reads each as it is written (the Python implementation only: `re` is Python's).
for _, source in cases:
    re.compile(source)

<a id="PYR-08"></a>
## PYR-08 · Evaluation refuses each of its kinds by the RegExp dialect's name

In [ ]:
# Evaluation (Q11): Python3's evaluator evaluates none of its kinds, and refuses each by the RegExp dialect's name; a
# regular expression is matched by `re`, a library call, not evaluated as an expression.
for spec, tag in [(alts(seq(a)), "pattern"), (seq(a), "alternative"), (group(seq(a)), "group"), (capture(seq(a)), "capturinggroup"),
                  (repeat(a, 0), "quantifier"), (klass(a), "characterclass"), (span(a, b), "characterclassrange"),
                  (charset("any"), "characterset"), (a, "character"), (check("start"), "assertion"), (ref(1), "backreference"),
                  (cond(1, seq(a)), "conditional"), (atomic(seq(a)), "atomicgroup"), (flags("i"), "flags"), (comment("c"), "comment")]:
    with raises(NotImplementedError, match=f"Python RegExp {tag} is not evaluated"):
        PV.OfAny(regexp(spec))

<a id="PYR-09"></a>
## PYR-09 · Completeness: each code `re`'s parser writes, 3.0 to 3.15, kept as data, is read as its kinds or listed as the compiler's, and each of its kinds is one of them; checked against the running Python's `re`

In [ ]:
# Completeness (Q9): `grammars/python3-re.json` lists the codes `re`'s parser writes, its operation, position and
# category codes and its flags, 3.0 to 3.15, each read as kinds of the RegExp dialect, with the attributes it gives
# them, or listed as not an expression, the compiler's, with its reason; every kind is one of them's, but those no code
# writes, each listed with its reason.
grammar = json.loads(Path("../../grammars/python3-re.json").read_text(encoding="utf-8"))
assert list(grammar) == ["grammar", "checked", "codes", "unwritten"]
CLASSES = ("OPCODES", "ATCODES", "CHCODES", "FLAGS")
ATTRIBUTES = {"assertionKind": R.ASSERTIONS, "setKind": R.SETS, "negate": (True, False), "greedy": (False,), "possessive": (True,),
              "letter": tuple(R.FLAG_LETTERS)}
minor = lambda text: int(text.split(".")[1])  # noqa: E731
mapped = set()
for code in grammar["codes"]:  # each read as kinds, with attributes, or not an expression, with its reason
    assert code["class"] in CLASSES and code["since"] in [f"3.{minor}" for minor in range(16)], code
    assert "until" not in code or minor(code["since"]) <= minor(code["until"]) < 15, code
    if "kinds" in code:
        assert "not" not in code and code["kinds"] and set(code["kinds"]) <= set(R.TAGS), code
        assert all(value in ATTRIBUTES[name] for name, value in code.get("attributes", {}).items()), code
        mapped |= set(code["kinds"])
    else:
        assert type(code["not"]) is str and code["not"] and "attributes" not in code, code
unwritten = {entry["kind"]: entry["reason"] for entry in grammar["unwritten"]}
assert set(unwritten) == {"pattern", "comment"} and not set(unwritten) & mapped and mapped | set(unwritten) == set(R.TAGS)
read = lambda name: {(c["code"], tuple(sorted(c.get("attributes", {}).items()))) for c in grammar["codes"] if name in c.get("kinds", ())}  # noqa: E731
assert {dict(attributes)["assertionKind"] for code, attributes in read("assertion") if "assertionKind" in dict(attributes)} == set(R.ASSERTIONS) - set(R.LOOKAROUNDS)
assert {dict(attributes)["setKind"] for code, attributes in read("characterset") if "setKind" in dict(attributes)} == set(R.SETS)
assert {dict(attributes)["letter"] for code, attributes in read("flags") if "letter" in dict(attributes)} == set(R.FLAG_LETTERS)
assert {code for code, _ in read("quantifier")} == {"MIN_REPEAT", "MAX_REPEAT", "POSSESSIVE_REPEAT"}
assert {c["code"]: c["since"] for c in grammar["codes"] if "atomicgroup" in c.get("kinds", ())} == {"ATOMIC_GROUP": "3.11"}
assert len(grammar["codes"]) == 86 and len(mapped) == 13
# The running Python's `re` against the data (the Python implementation only: `re` is Python's): every code its
# `re._constants` defines is listed, for its version, and every one listed it defines.
from re import _constants as constants  # noqa: E402

version = sys.version_info[:2]
live = {code["code"] for code in grammar["codes"] if minor(code["since"]) <= version[1] <= minor(code.get("until", "3.15"))}
defined = {code.name for group_ in (constants.OPCODES, constants.ATCODES, constants.CHCODES) for code in group_}
defined |= {"MIN_REPEAT", "MAX_REPEAT"} | {name for name in dir(constants) if name.startswith("SRE_FLAG_")}
assert live == defined, (live ^ defined)